# GOES Daily Evolution Viewer

Inspect how GOES-R FDCF active fire mask and FRP evolve within a single day for one TS-SatFire event.

This notebook reprojects every GOES frame for a selected date to the same `256x256` FirePred/VIIRS crop grid, then shows:

- active pixel count over time
- FRP sum/max over time
- sampled active mask frames across the day
- sampled FRP frames across the day
- daily aggregate maps: active frequency, FRP sum, FRP max

For next-day prediction, use only frames from the allowed input day `d`; do not use target day `d+1`.

In [ ]:
from pathlib import Path
import os
import sys
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.enums import Resampling

REPO_ROOT = Path.cwd()
LEGACY_ROOT = REPO_ROOT / 'legacy'
if str(LEGACY_ROOT) not in sys.path:
    sys.path.insert(0, str(LEGACY_ROOT))

from scripts.analyze_pred_event_windows import load_daily_masks
from scripts.dataset_gen_pred_goes_spatial import (
    FIRE_MASK_CODES,
    collect_goes_files_by_day,
    crop_profile,
    daily_maps,
    find_event_dir,
    firepred_path_from_viirs,
    parse_timestamp_from_name,
    reproject_to_crop,
    viirs_day_files,
)

GOES_ROOT = Path(os.environ.get('TS_SATFIRE_GOES_ROOT', '/home/jlc3q/data/GOES_clipped_tif_common_wgs84'))
CANDIDATE_ROOT = Path(os.environ.get('TS_SATFIRE_EVENT_CANDIDATE_ROOT', '/home/jlc3q/data/SatFire/event_candidates'))
SPLIT = os.environ.get('TS_SATFIRE_OVERLAP_SPLIT', 'test')

print('repo:', REPO_ROOT)
print('GOES root:', GOES_ROOT)
print('candidate root:', CANDIDATE_ROOT)

## Select Fire and Date

Default picks the same style of useful positive example from the candidate table. You can override `FIRE_ID` and `DATE` manually.

In [ ]:
FIRE_ID = None
DATE = None

candidate_file = CANDIDATE_ROOT / f'pred_event_candidates_{SPLIT}_conn8_r5p0_mincomp1.csv'
assert candidate_file.exists(), candidate_file

if FIRE_ID is None or DATE is None:
    usecols = ['fire_id', 'date', 'next_date', 'component_id', 'label_ignited_next_day']
    df = pd.read_csv(candidate_file, usecols=usecols)
    pos = df[df.label_ignited_next_day == 1]
    key = pos.groupby(['fire_id', 'date', 'next_date']).size().sort_values(ascending=False).index[0]
    FIRE_ID, DATE, NEXT_DATE = key
else:
    sub = pd.read_csv(candidate_file, usecols=['fire_id', 'date', 'next_date'])
    hit = sub[(sub.fire_id == FIRE_ID) & (sub.date == DATE)]
    NEXT_DATE = hit.next_date.iloc[0] if len(hit) else None

print('selected fire:', FIRE_ID)
print('selected date:', DATE)
print('next date:', NEXT_DATE)

## Load GOES Frames for the Selected Day

Each GOES frame is reprojected to the FirePred/VIIRS crop grid. `mask` is converted to active fire using FDCF fire mask codes; `frp` is clipped to positive values.

In [ ]:
files = viirs_day_files(FIRE_ID)
assert files, FIRE_ID
ref_firepred = firepred_path_from_viirs(files[0])
dst_profile = crop_profile(ref_firepred)

event_dir = find_event_dir(GOES_ROOT, FIRE_ID)
assert event_dir is not None, f'No GOES event dir found for {FIRE_ID}'
goes_by_day = collect_goes_files_by_day(event_dir)
bucket = goes_by_day.get(DATE, {'mask': [], 'frp': []})
print('GOES event dir:', event_dir)
print('mask frames:', len(bucket['mask']))
print('frp frames:', len(bucket['frp']))
assert bucket['mask'], f'No mask frames for {DATE}'

mask_by_ts = {parse_timestamp_from_name(p.name): p for p in bucket['mask']}
frp_by_ts = {parse_timestamp_from_name(p.name): p for p in bucket['frp']}
timestamps = sorted(ts for ts in mask_by_ts if ts is not None)
print('first timestamp:', timestamps[0])
print('last timestamp:', timestamps[-1])
print('n timestamps:', len(timestamps))

In [ ]:
def load_projected_frame(ts):
    mask_path = mask_by_ts.get(ts)
    frp_path = frp_by_ts.get(ts)
    active = np.zeros((256, 256), dtype=np.float32)
    frp = np.zeros((256, 256), dtype=np.float32)

    if mask_path is not None:
        with rasterio.open(mask_path) as src:
            arr = np.nan_to_num(src.read(1), nan=0.0)
            active_native = np.isin(arr, list(FIRE_MASK_CODES)).astype(np.float32)
            active = reproject_to_crop(active_native, src.profile, dst_profile, Resampling.nearest)
            active = (active > 0).astype(np.float32)

    if frp_path is not None:
        with rasterio.open(frp_path) as src:
            arr = np.nan_to_num(src.read(1), nan=0.0, posinf=0.0, neginf=0.0)
            arr = np.where(arr > 0, arr, 0).astype(np.float32)
            frp = reproject_to_crop(arr, src.profile, dst_profile, Resampling.bilinear)
            frp = np.where(frp > 0, frp, 0).astype(np.float32)

    return active, frp

records = []
active_frames = []
frp_frames = []

for ts in timestamps:
    active, frp = load_projected_frame(ts)
    active_frames.append(active)
    frp_frames.append(frp)
    records.append({
        'timestamp': ts,
        'active_pixels': int(active.sum()),
        'frp_sum': float(frp.sum()),
        'frp_max': float(frp.max()),
        'frp_active_mean': float(frp[active > 0].mean()) if np.any(active > 0) else 0.0,
    })

ts_df = pd.DataFrame(records)
active_stack = np.stack(active_frames, axis=0)
frp_stack = np.stack(frp_frames, axis=0)
print(ts_df.head())
print(ts_df.tail())
print('active stack:', active_stack.shape, 'frp stack:', frp_stack.shape)

## Time Series Within the Day

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), dpi=120, sharex=True)
axes[0].plot(ts_df.timestamp, ts_df.active_pixels, marker='.', linewidth=1)
axes[0].set_ylabel('active pixels')
axes[0].set_title(f'{FIRE_ID} GOES daily evolution on {DATE}')

axes[1].plot(ts_df.timestamp, ts_df.frp_sum, marker='.', linewidth=1, color='tab:orange')
axes[1].set_ylabel('FRP sum')

axes[2].plot(ts_df.timestamp, ts_df.frp_max, marker='.', linewidth=1, color='tab:red')
axes[2].set_ylabel('FRP max')
axes[2].set_xlabel('timestamp')

for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()

## Sampled Frames Across the Day

These panels sample up to 12 frames evenly across the day. Red mask panels show active fire pixels. FRP panels use a robust percentile stretch.

In [ ]:
def sample_indices(n, k=12):
    if n <= k:
        return np.arange(n)
    return np.linspace(0, n - 1, k).round().astype(int)

idxs = sample_indices(len(timestamps), 12)
fig, axes = plt.subplots(2, len(idxs), figsize=(2.3 * len(idxs), 5.2), dpi=120)
if len(idxs) == 1:
    axes = np.array(axes).reshape(2, 1)
frp_vmax = np.nanpercentile(frp_stack[frp_stack > 0], 99) if np.any(frp_stack > 0) else 1

for ax_col, idx in enumerate(idxs):
    ts = timestamps[idx]
    axes[0, ax_col].imshow(active_stack[idx], cmap='Reds', vmin=0, vmax=1)
    axes[0, ax_col].set_title(ts.strftime('%H:%M'))
    axes[0, ax_col].set_xticks([])
    axes[0, ax_col].set_yticks([])

    axes[1, ax_col].imshow(frp_stack[idx], cmap='inferno', vmin=0, vmax=frp_vmax)
    axes[1, ax_col].set_xticks([])
    axes[1, ax_col].set_yticks([])

axes[0, 0].set_ylabel('active')
axes[1, 0].set_ylabel('FRP')
plt.suptitle(f'Sampled GOES frames on {DATE}', y=1.03)
plt.tight_layout()

## Daily Aggregates

These are the daily maps we can use as candidate-level features for an end-of-day forecast.

In [ ]:
daily = daily_maps(DATE, goes_by_day, dst_profile)
active_frequency = daily['active_frequency']
frp_sum = daily['frp_sum_log1p']
frp_max = daily['frp_max_log1p']

daily_active_count = active_stack.sum(axis=0)

fig, axes = plt.subplots(1, 4, figsize=(18, 5), dpi=120)
items = [
    ('active count over day', daily_active_count, 'magma'),
    ('active frequency', active_frequency, 'magma'),
    ('FRP sum log1p', frp_sum, 'inferno'),
    ('FRP max log1p', frp_max, 'inferno'),
]
for ax, (title, img, cmap) in zip(axes, items):
    vmax = np.nanpercentile(img[img > 0], 99) if np.any(img > 0) else 1
    ax.imshow(img, cmap=cmap, vmin=0, vmax=max(float(vmax), 1e-6))
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

## VIIRS Context Overlay

Grey/red = current VIIRS cumulative fire at the selected date. Blue = next-day VIIRS growth. GOES maps shown here are still only from the selected input date.

In [ ]:
label_sel = 0 if SPLIT == 'test' else 1
dates, masks = load_daily_masks(FIRE_ID, label_sel)
day_idx = dates.index(DATE)
current_mask = masks[day_idx]
next_mask = masks[day_idx + 1]
growth_mask = next_mask & ~current_mask

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=120)
for ax, (title, img, cmap) in zip(
    axes,
    [
        ('GOES active frequency + VIIRS current/growth', active_frequency, 'magma'),
        ('GOES FRP sum + VIIRS current/growth', frp_sum, 'inferno'),
        ('GOES FRP max + VIIRS current/growth', frp_max, 'inferno'),
    ],
):
    vmax = np.nanpercentile(img[img > 0], 99) if np.any(img > 0) else 1
    ax.imshow(img, cmap=cmap, vmin=0, vmax=max(float(vmax), 1e-6))
    ax.imshow(np.where(current_mask, 1, np.nan), cmap='Greys', alpha=0.35, vmin=0, vmax=1)
    ax.imshow(np.where(growth_mask, 1, np.nan), cmap='Blues', alpha=0.8, vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()

print('VIIRS current pixels:', int(current_mask.sum()))
print('VIIRS next-day growth pixels:', int(growth_mask.sum()))

## Notes

If FRP frames show a clearer moving core than active masks, the next candidate-level features should focus on:

- FRP local max/mean around each candidate
- distance to high-FRP core
- component-to-FRP-centroid direction
- candidate direction alignment with the FRP centroid or FRP-weighted motion axis